# DSCI 511 Final Project:

## Research Question
**Which car performance and race strategy factors best predict a driver's final race finishing position?**

We collect telemetry and performance data from the [OpenF1 API](https://openf1.org/) across all 2023 Formula 1 race sessions, then combine it with historical race results from [StatsF1](https://www.statsf1.com) to build a unified dataset.

## OpenF1 Endpoints

| Endpoint | Description |
|---|---|
| `car_data` | High-frequency telemetry (speed, throttle, brake, RPM, gear, DRS) |
| `sessions` | Session metadata (race name, circuit, date) |
| `drivers` | Driver info per session |
| `laps` | Lap-by-lap timing data |
| `stints` | Tire stint info (compound, lap range) |
| `pit` | Pit stop events (lap number, duration) |
| `position` | Track position throughout session (aggregated to starting position) |
| `session_result` | Final classified results and grid positions |

## Sample Request

In [1]:
import requests
import pandas as pd

response = requests.get(
  "https://api.openf1.org/v1/car_data",
  params={"driver_number": 1, "session_key": 9158}
)

df = pd.DataFrame(response.json())
print(f"Rows returned: {len(df)}")
df.head(10)

Rows returned: 18026


,date,session_key,throttle,speed,brake,rpm,meeting_key,driver_number,n_gear,drs
0,2023-09-15T09:15:02.731000+00:00,9158,0,0,0,0,1219,1,0,9
1,2023-09-15T09:15:02.971000+00:00,9158,0,0,0,0,1219,1,0,9
2,2023-09-15T09:15:03.211000+00:00,9158,0,0,0,0,1219,1,0,9
3,2023-09-15T09:15:03.491000+00:00,9158,0,0,0,0,1219,1,0,9
4,2023-09-15T09:15:03.771000+00:00,9158,0,0,0,0,1219,1,0,9
5,2023-09-15T09:15:04.092000+00:00,9158,0,0,0,0,1219,1,0,9
6,2023-09-15T09:15:04.492000+00:00,9158,0,0,0,0,1219,1,0,9
7,2023-09-15T09:15:04.732000+00:00,9158,0,0,0,0,1219,1,0,9
8,2023-09-15T09:15:05.012000+00:00,9158,0,0,0,0,1219,1,0,9
9,2023-09-15T09:15:05.292000+00:00,9158,0,0,0,0,1219,1,0,9


Initially, we started with setting the `SESSION_KEY = 9158` but after conducting some research, doing so would limit us to a single race, which is not enough data to answer our propsed research question "Which features best predict final positions?". We would only have one data point per driver (20 rows total). We would need more variation across many races: different circuits, weather conditions, strategies, etc. The solution for now is to keep `SESSION_KEY` as a toggle while developing/testing. When `SESSION_KEY` is set we retrieve one session; when it is `None` we retrieve all race sessions for the year.

In [2]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.openf1.org/v1"
YEAR = 2023
SESSION_KEY = None  # Set to None to collect all 2023 races

session_params = {"session_key": SESSION_KEY} if SESSION_KEY else {"year": YEAR}

Before collecting data, we list all race sessions for the year to identify valid `session_key` values. This is useful for picking a specific session to test with, or to verify which sessions OpenF1 has coverage for.
The `sessions` endpoint returns metadata for each race session including circuit name, country, and date. This forms the backbone of our dataset, providing the race context that all other endpoints are joined against. 

In [3]:
all_sessions = []
for year in [2023, 2024, 2025]:
    r = requests.get("https://api.openf1.org/v1/sessions", params={"year": year})
    all_sessions.extend(r.json())

df_sessions = pd.DataFrame(all_sessions)
session_keys = df_sessions["session_key"].tolist()

print(f"All sessions from 2023-2025: {len(df_sessions)}")
df_sessions[["session_key", "session_name", "date_start", "circuit_short_name", "country_name", "year"]].sample(10)

All sessions from 2023-2025: 364


,session_key,session_name,date_start,circuit_short_name,country_name,year
247,9689,Qualifying,2025-03-15T05:00:00+00:00,Melbourne,Australia,2025
338,9888,Race,2025-10-19T19:00:00+00:00,Austin,United States,2025
0,9222,Day 1,2023-02-23T07:00:00+00:00,Sakhir,Bahrain,2023
280,9973,Practice 2,2025-05-23T15:00:00+00:00,Monte Carlo,Monaco,2025
332,9892,Qualifying,2025-10-04T13:00:00+00:00,Singapore,Singapore,2025
170,9539,Race,2024-06-23T13:00:00+00:00,Catalunya,Spain,2024
92,9221,Race,2023-10-08T17:00:00+00:00,Lusail,Qatar,2023
105,9308,Sprint Qualifying,2023-11-04T14:00:00+00:00,Interlagos,Brazil,2023
32,9086,Race,2023-05-21T13:00:00+00:00,Imola,Italy,2023
53,9119,Practice 1,2023-07-07T11:30:00+00:00,Silverstone,United Kingdom,2023


The `drivers` endpoint returns the list of drivers who participated in each session, along with their team and nationality. We deduplicate by `driver_number` and `session_key` since the raw API response can contain repeated entries for the same driver.

In [4]:
all_drivers = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/drivers", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_drivers.extend(data)


df_drivers = pd.DataFrame(all_drivers).drop_duplicates(subset=["driver_number", "session_key"])
print(f"Driver records: {len(df_drivers)}")
df_drivers[["session_key", "driver_number", "full_name", "team_name", "country_code"]].head(10)

Driver records: 2883


,session_key,driver_number,full_name,team_name,country_code
0,9222,1,Max VERSTAPPEN,Red Bull Racing,NED
1,9222,2,Logan SARGEANT,Williams,USA
2,9222,4,Lando NORRIS,McLaren,GBR
3,9222,10,Pierre GASLY,Alpine,FRA
4,9222,14,Fernando ALONSO,Aston Martin,ESP
5,9222,16,Charles LECLERC,Ferrari,MON
6,9222,20,Kevin MAGNUSSEN,Haas F1 Team,DEN
7,9222,21,Nyck DE VRIES,AlphaTauri,NED
8,9222,22,Yuki TSUNODA,AlphaTauri,JPN
9,9222,23,Alexander ALBON,Williams,THA


The `laps` endpoint returns lap-by-lap timing data for every driver in a session, including lap duration, sector times, and tire compound. In the final dataset these are aggregated into per-driver summary statistics: average lap time, fastest lap, and lap time standard deviation (a measure of consistency).

In [5]:
all_laps = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/laps", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_laps.extend(data)

df_laps = pd.DataFrame(all_laps)
print(f"Total lap records: {len(df_laps)}")
df_laps.head(5)

Total lap records: 79411


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed
0,1141,7768,22,1,None,NaN,52.812,25.206,139.0,198.0,True,NaN,"[2064, 2064, 2064, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2051, 2051, 2051, 204...","[2051, 2051, 2051, 2051, 2049, 2051]",134.0
1,1141,7768,21,1,2023-03-04T15:00:05.921000+00:00,40.779,55.481,25.847,166.0,194.0,True,122.107,"[2064, 2064, 2064, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2051, 2049]",162.0
2,1141,7768,16,1,2023-03-04T15:00:32.592000+00:00,35.378,49.509,26.893,208.0,223.0,True,111.780,"[2064, 2064, 2064, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2051, 2051, 2049, 2049, 2049, 204...","[2051, 2049, 2049, 2049, 2049, 2049]",189.0
3,1141,7768,55,1,2023-03-04T15:00:35.342000+00:00,37.209,50.194,28.751,205.0,219.0,True,116.154,"[2064, 2064, 2064, 2049, 2049, 2049, 2049, 204...","[2051, 2051, 2049, 2049, 2049, 2049, 2049, 205...","[2049, 2049, 2049, 2049, 2049, 2049]",158.0
4,1141,7768,27,1,2023-03-04T15:01:18.810000+00:00,36.756,57.378,28.942,169.0,152.0,True,123.076,"[2064, 2064, 2064, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049]",181.0


A stint is a continuous run on one set of tires between pit stops. The `stints` endpoint tells us which tire compound each driver used, how many laps they ran on it, and how many stints they completed in total.

In [6]:
all_stints = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/stints", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_stints.extend(data)

df_stints = pd.DataFrame(all_stints)
print(f"Total stint records: {len(df_stints)}")
df_stints.head(5)

Total stint records: 9437


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
0,1210,9087,1,11,1.0,4.0,HARD,0
1,1210,9087,1,77,1.0,15.0,HARD,0
2,1210,9087,1,21,1.0,17.0,HARD,0
3,1210,9087,1,22,1.0,18.0,HARD,0
4,1210,9087,1,24,1.0,11.0,HARD,0


The `pit` endpoint records each pit stop event, including the lap number and time spent in the pit lane. OpenF1's coverage of pit stop data is incomplete for some sessions — those return a 404 and are silently skipped by the `isinstance` check, so the loop is safe to run across all sessions.

In [7]:
all_pit = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/pit", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_pit.extend(data)

df_pit = pd.DataFrame(all_pit)
print(f"Total pit stop records: {len(df_pit)}")
df_pit.head(5)

Total pit stop records: 7971


,date,session_key,stop_duration,meeting_key,driver_number,lap_number,lane_duration,pit_duration
0,2023-09-24T05:07:12.968000+00:00,9173,NaN,1220,31,1,27.0,27.0
1,2023-09-24T05:07:15.655000+00:00,9173,NaN,1220,24,1,41.0,41.0
2,2023-09-24T05:07:19.952000+00:00,9173,NaN,1220,23,1,50.1,50.1
3,2023-09-24T05:07:54.608000+00:00,9173,NaN,1220,77,1,55.8,55.8
4,2023-09-24T05:09:29.028000+00:00,9173,NaN,1220,11,2,32.4,32.4


The `position` endpoint tracks each driver's track position at sub-second frequency throughout the session, which is similar in volume to `car_data`. To make it usable in the final flat dataset, we aggregate immediately inside the loop by taking the first recorded position per driver per session, which approximates their starting grid position.

In [8]:
# Position data is sub-second frequency like car_data, so we aggregate immediately.
# We capture the starting position (first recorded position per driver per session).
all_position_agg = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/position", 
        params={"session_key": session_key}
    )
    data = response.json()
    df_pos = pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()
    if not df_pos.empty:
        start_pos = (
            df_pos.sort_values("date")
            .groupby("driver_number")
            .first()[["position"]]
            .rename(columns={"position": "start_position"})
            .reset_index()
        )
        start_pos["session_key"] = session_key
        all_position_agg.append(start_pos)

df_position = pd.concat(all_position_agg, ignore_index=True) if all_position_agg else pd.DataFrame()
print(f"Position records (aggregated to start position per driver): {len(df_position)}")
df_position.head(10)

Position records (aggregated to start position per driver): 2508


,driver_number,start_position,session_key
0,1,1,9222
1,2,2,9222
2,4,3,9222
3,10,4,9222
4,14,5,9222
5,16,6,9222
6,20,7,9222
7,21,8,9222
8,22,9,9222
9,23,10,9222


The `session_result` endpoint provides the official final classification for each session which are finishing position, championship points, and starting grid position. If this was a complete data science project, this would be the **target variable** for our research question: we aim to predict a driver's finishing `position` using the features collected from all other endpoints. It also serves as the base of the final merge, giving us one row per driver per race.

In [9]:
all_results = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/session_result", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_results.extend(data)

df_session_result = pd.DataFrame(all_results)
print(f"Total result records: {len(df_session_result)}")
df_session_result.head(10)

Total result records: 2276


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,session_key
0,1.0,1,57.0,26.0,False,False,False,5258.241,0,1208,9078
1,2.0,11,57.0,18.0,False,False,False,5263.625,5.384,1208,9078
2,3.0,14,57.0,15.0,False,False,False,5284.546,26.305,1208,9078
3,4.0,63,57.0,12.0,False,False,False,5291.47,33.229,1208,9078
4,5.0,55,57.0,10.0,False,False,False,5300.752,42.511,1208,9078
5,6.0,44,57.0,8.0,False,False,False,5309.49,51.249,1208,9078
6,7.0,16,57.0,6.0,False,False,False,5311.229,52.988,1208,9078
7,8.0,10,57.0,4.0,False,False,False,5313.911,55.67,1208,9078
8,9.0,31,57.0,2.0,False,False,False,5316.364,58.123,1208,9078
9,10.0,20,57.0,1.0,False,False,False,5321.186,62.945,1208,9078


The `car_data` endpoint returns high-frequency telemetry: speed, throttle, brake, RPM, gear, and DRS that were sampled multiple times per second. It is the heaviest endpoint with 18k+ rows per driver per session. Retrieving all its data during the full 2023 season without limiting any parameter would result in ~440 API calls, which might take over an hour. The proposed solution for now is instead of all 20 drivers, we only pick the top finishers (positions 1-5) to reduce the calls by 75%. Therefore, when we aggregate all the data in the end, drivers without `car_data` (positions 6-20) will get `Nan` for the `car_data` columns.

In [10]:
SAMPLE_DRIVERS = 5  # number of top finishers to fetch car_data for per session

all_car_data = []
for session_key in df_sessions["session_key"]:
    top_drivers = (
        df_session_result[df_session_result["session_key"] == session_key]
        .sort_values("position")
        .head(SAMPLE_DRIVERS)["driver_number"]
    )
    for driver_number in top_drivers:
        response = requests.get(
            f"{BASE_URL}/car_data", 
            params={"session_key": session_key, "driver_number": driver_number}
        )
        data = response.json()
        if isinstance(data, list):
            all_car_data.extend(data)

df_car_data = pd.DataFrame(all_car_data)
print(f"Total car data records: {len(df_car_data)}")
df_car_data.head(10)

Total car data records: 13921293


,date,session_key,throttle,n_gear,meeting_key,speed,rpm,driver_number,brake,drs
0,2023-05-07T18:31:02.856000+00:00,9078,104,0,1208,0,634,14,104,1
1,2023-05-07T18:31:03.215000+00:00,9078,104,0,1208,0,634,14,104,1
2,2023-05-07T18:31:03.415000+00:00,9078,104,0,1208,0,634,14,104,1
3,2023-05-07T18:31:03.695000+00:00,9078,104,0,1208,0,634,14,104,1
4,2023-05-07T18:31:03.895000+00:00,9078,104,0,1208,0,634,14,104,1
5,2023-05-07T18:31:04.255000+00:00,9078,104,0,1208,0,634,14,104,1
6,2023-05-07T18:31:04.495000+00:00,9078,104,0,1208,0,634,14,104,1
7,2023-05-07T18:31:04.655000+00:00,9078,104,0,1208,0,634,14,104,1
8,2023-05-07T18:31:04.975000+00:00,9078,104,0,1208,0,634,14,104,1
9,2023-05-07T18:31:05.335000+00:00,9078,104,0,1208,0,634,14,104,1


With all endpoints collected, we can build the final dataset. Each high-frequency source (`laps`, `pit`, `stints`, `car_data`) is first aggregated down to one summary row per driver per session, then joined together using `session_key` and `driver_number` as keys.

The base of every merge is `df_session_result` which has one row per driver per race. All other data is joined with left joins, so drivers with missing data for a given endpoint simply receive `NaN` rather than being dropped. The result is a flat dataset where each row represents one driver's complete performance profile for a single race, ready for exploratory analysis and predictive modelling.

In [ ]:
# Aggregate laps per driver per session
df_laps_agg = (
    df_laps.groupby(["session_key", "driver_number"])
    .agg(
        total_laps=("lap_number", "max"),
        avg_lap_duration=("lap_duration", "mean"),
        fastest_lap=("lap_duration", "min"),
        lap_time_std=("lap_duration", "std"),
    )
    .reset_index()
)

# Aggregate pit stops per driver per session
df_pit_agg = (
    df_pit.groupby(["session_key", "driver_number"])
    .agg(
        pit_stop_count=("pit_duration", "count"),
        total_pit_time=("pit_duration", "sum"),
        avg_pit_duration=("pit_duration", "mean"),
    )
    .reset_index()
)

# Aggregate stints per driver per session
df_stints_agg = (
    df_stints.groupby(["session_key", "driver_number"])
    .agg(
        stint_count=("stint_number", "max"),
        compounds_used=("compound", lambda x: list(x.unique())),
    )
    .reset_index()
)

# Aggregate car_data per driver per session
df_car_data_agg = (
    df_car_data.groupby(["session_key", "driver_number"])
    .agg(
        avg_speed=("speed", "mean"),
        max_speed=("speed", "max"),
        avg_throttle=("throttle", "mean"),
        avg_brake=("brake", "mean"),
        avg_rpm=("rpm", "mean"),
        drs_usage=("drs", lambda x: (x >= 10).mean()),
    )
    .reset_index()
)

# Build combined dataset — one row per driver per race
df_combined = (
    df_session_result
    .merge(df_sessions[["session_key", "session_name", "circuit_short_name", "country_name", "date_start"]], on="session_key", how="left")
    .merge(df_drivers[["session_key", "driver_number", "full_name", "team_name", "country_code"]], on=["session_key", "driver_number"], how="left")
    .merge(df_laps_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_pit_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_stints_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_position[["session_key", "driver_number", "start_position"]], on=["session_key", "driver_number"], how="left")
    .merge(df_car_data_agg, on=["session_key", "driver_number"], how="left")
)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Columns: {list(df_combined.columns)}")
df_combined.head(10)


Combined dataset shape: (2276, 34)
Columns: ['position', 'driver_number', 'number_of_laps', 'points', 'dnf', 'dns', 'dsq', 'duration', 'gap_to_leader', 'meeting_key', 'session_key', 'session_name', 'circuit_short_name', 'country_name', 'date_start', 'full_name', 'team_name', 'country_code', 'total_laps', 'avg_lap_duration', 'fastest_lap', 'lap_time_std', 'pit_stop_count', 'total_pit_time', 'avg_pit_duration', 'stint_count', 'compounds_used', 'start_position', 'avg_speed', 'max_speed', 'avg_throttle', 'avg_brake', 'avg_rpm', 'drs_usage']


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,...,avg_pit_duration,stint_count,compounds_used,start_position,avg_speed,max_speed,avg_throttle,avg_brake,avg_rpm,drs_usage
0,1.0,1,57.0,26.0,False,False,False,5258.241,0,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,11,57.0,18.0,False,False,False,5263.625,5.384,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,14,57.0,15.0,False,False,False,5284.546,26.305,1208,...,NaN,NaN,NaN,NaN,126.421152,333.0,45.358393,19.366473,6394.527482,0.007281
3,4.0,63,57.0,12.0,False,False,False,5291.47,33.229,1208,...,NaN,NaN,NaN,NaN,128.663650,344.0,47.658482,19.851412,6456.866830,0.024869
4,5.0,55,57.0,10.0,False,False,False,5300.752,42.511,1208,...,NaN,NaN,NaN,NaN,128.665334,341.0,48.735117,22.308540,6509.830768,0.038603
5,6.0,44,57.0,8.0,False,False,False,5309.49,51.249,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7.0,16,57.0,6.0,False,False,False,5311.229,52.988,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8.0,10,57.0,4.0,False,False,False,5313.911,55.67,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.0,31,57.0,2.0,False,False,False,5316.364,58.123,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10.0,20,57.0,1.0,False,False,False,5321.186,62.945,1208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
df_combined.sample(10)

,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,...,avg_pit_duration,stint_count,compounds_used,start_position,avg_speed,max_speed,avg_throttle,avg_brake,avg_rpm,drs_usage
1826,9.0,14,66.0,2.0,False,False,False,5598.939,21.564,1262,...,NaN,4.0,"[SOFT, MEDIUM]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1545,8.0,10,18.0,NaN,False,False,False,"[77.149, 77.048, 76.892]","[0.644, 0.747, 0.946]",1248,...,NaN,7.0,[SOFT],NaN,NaN,NaN,NaN,NaN,NaN,NaN
1684,7.0,30,33.0,NaN,False,False,False,71.814,0.861,1261,...,123.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
870,1.0,4,23.0,NaN,False,False,False,78.564,0,1231,...,NaN,5.0,"[MEDIUM, SOFT]",NaN,88.674909,319.0,55.090484,36.878648,4828.433726,0.043965
1964,9.0,14,31.0,NaN,False,False,False,65.457,0.877,1264,...,NaN,5.0,"[MEDIUM, SOFT]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2267,12.0,27,31.0,NaN,False,False,False,76.68,1.056,1266,...,NaN,7.0,"[MEDIUM, SOFT]",12.0,NaN,NaN,NaN,NaN,NaN,NaN
2030,15.0,43,69.0,0.0,False,False,False,None,+1 LAP,1264,...,NaN,3.0,"[SOFT, MEDIUM, HARD]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
697,8.0,18,26.0,NaN,False,False,False,90.891,0.517,1229,...,NaN,3.0,[SOFT],NaN,NaN,NaN,NaN,NaN,NaN,NaN
361,5.0,55,18.0,NaN,False,False,False,"[66.187, 65.434, 65.136]","[0.0, 0.063, 0.696]",1213,...,NaN,6.0,"[SOFT, MEDIUM]",NaN,74.028342,325.0,33.982062,22.115807,4374.529239,0.072182
381,5.0,14,24.0,4.0,False,False,False,1856.839,30.109,1213,...,NaN,1.0,[INTERMEDIATE],NaN,91.080870,315.0,27.338783,11.458783,4820.801623,0.033739


## Web Scraping: StatsF1

Our second data source is [StatsF1](https://www.statsf1.com), which publishes structured race result tables for every Grand Prix. Each race has a `/classement.aspx` sub-page with columns for finishing position, driver, team, laps completed, time/gap, and points.

We scrape all 22 races from the 2023 season, then merge the results into `df_combined` using round number and driver number as join keys. This adds historical race context (e.g. laps completed, gap to winner) that complements the telemetry and timing data from OpenF1.

In [49]:
BASE_STATSF1 = "https://www.statsf1.com"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# All 22 race slugs for 2023 in calendar order
RACE_SLUGS_2023 = [
    "bahrein", "arabie-saoudite", "australie", "azerbaidjan", "miami",
    "monaco", "espagne", "canada", "autriche", "grande-bretagne",
    "hongrie", "belgique", "pays-bas", "italie", "singapour",
    "japon", "qatar", "etats-unis", "mexico-city", "sao-paulo",
    "las-vegas", "abou-dhabi"
]

all_statsf1 = []
for round_num, slug in enumerate(RACE_SLUGS_2023, start=1):
    url = f"{BASE_STATSF1}/en/{YEAR}/{slug}/classement.aspx"
    r = requests.get(url, headers=HEADERS)
    try:
        tables = pd.read_html(r.text)
        if tables:
            df_race = tables[0]
            df_race["race_slug"] = slug
            df_race["round"] = round_num
            all_statsf1.append(df_race)
    except Exception as e:
        print(f"Could not parse {slug}: {e}")

df_statsf1_raw = pd.concat(all_statsf1, ignore_index=True)
print(f"Total StatsF1 records: {len(df_statsf1_raw)}")
print(f"Columns: {list(df_statsf1_raw.columns)}")
df_statsf1_raw.head(10)

C:\Users\Tin\AppData\Local\Temp\ipykernel_18336\13151577.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)
C:\Users\Tin\AppData\Local\Temp\ipykernel_18336\13151577.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)
C:\Users\Tin\AppData\Local\Temp\ipykernel_18336\13151577.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)
C:\Users\Tin\AppData\Local\Temp\ipykernel_18336\13151577.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap

Total StatsF1 records: 468
Columns: ['Pos', 'N°', 'Driver', 'Chassis', 'Engine', 'Lap', 'Unnamed: 6', 'Pts', 'race_slug', 'round']


C:\Users\Tin\AppData\Local\Temp\ipykernel_18336\13151577.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


,Pos,N°,Driver,Chassis,Engine,Lap,Unnamed: 6,Pts,race_slug,round
0,1,1.0,Max VERSTAPPEN,Red Bull,Honda RBPT,57.0,1h 33m 56.736s ( 196.862 km/h ),25.0,bahrein,1
1,2,11.0,Sergio PEREZ,Red Bull,Honda RBPT,57.0,1h 34m 08.723s ( +11.987s ),18.0,bahrein,1
2,3,14.0,Fernando ALONSO,Aston Martin,Mercedes,57.0,1h 34m 35.373s ( +38.637s ),15.0,bahrein,1
3,4,55.0,Carlos SAINZ,Ferrari,Ferrari,57.0,1h 34m 44.788s ( +48.052s ),12.0,bahrein,1
4,5,44.0,Lewis HAMILTON,Mercedes,Mercedes,57.0,1h 34m 47.713s ( +50.977s ),10.0,bahrein,1
5,6,18.0,Lance STROLL,Aston Martin,Mercedes,57.0,1h 34m 51.238s ( +54.502s ),8.0,bahrein,1
6,7,63.0,George RUSSELL,Mercedes,Mercedes,57.0,1h 34m 52.609s ( +55.873s ),6.0,bahrein,1
7,8,77.0,Valtteri BOTTAS,Alfa Romeo,Ferrari,57.0,1h 35m 09.383s ( +1m 12.647s ),4.0,bahrein,1
8,9,10.0,Pierre GASLY,Alpine,Renault,57.0,1h 35m 10.489s ( +1m 13.753s ),2.0,bahrein,1
9,10,23.0,Alexander ALBON,Williams,Mercedes,57.0,1h 35m 26.510s ( +1m 29.774s ),1.0,bahrein,1


We rename the scraped columns to consistent names, then match each StatsF1 row to OpenF1's `driver_number` and `session_key`.

Driver names match exactly between the two sources (both use "First LAST" format with the last name in capitals), so we can map directly using `df_drivers`. For the race-to-session mapping we assign round numbers 1–22 to both the StatsF1 slugs (already in calendar order) and the OpenF1 race sessions (sorted by date, excluding sprint sessions).

In [34]:
# Standardize column names (actual names depend on pd.read_html output — adjust if needed)
col_mapping = {
    "Pos": "finish_pos_statsf1",
    "N°": "car_number_statsf1",
    "Driver": "driver_name_statsf1",
    "Chassis": "team_statsf1",
    "Engine": "engine_statsf1",
    "Lap": "laps_statsf1",
    "Unnamed: 6": "time_statsf1",
    "Pts": "points_statsf1",
}
df_statsf1 = df_statsf1_raw.rename(columns={k: v for k, v in col_mapping.items() if k in df_statsf1_raw.columns})

# Map driver name → driver_number using df_drivers (names match: "Carlos SAINZ" in both)
name_to_number = (
    df_drivers[["driver_number", "full_name"]]
    .drop_duplicates("full_name")
    .set_index("full_name")["driver_number"]
    .to_dict()
)
df_statsf1["driver_number"] = df_statsf1["driver_name_statsf1"].map(name_to_number)

unmatched = df_statsf1[df_statsf1["driver_number"].isna()]["driver_name_statsf1"].unique()
if len(unmatched):
    print(f"Unmatched driver names (need manual fix): {unmatched}")

# Assign round numbers to OpenF1 race sessions (exclude sprints, sort by date)
df_races_ordered = (
    df_sessions[df_sessions["session_name"] == "Race"]
    .sort_values("date_start")
    .reset_index(drop=True)
    .assign(round=lambda df: df.index + 1)
)

# Map round to session_key
df_statsf1 = df_statsf1.merge(df_races_ordered[["round", "session_key"]], on="round", how="left")

print(f"StatsF1 records matched to session_key: {df_statsf1['session_key'].notna().sum()} / {len(df_statsf1)}")
df_statsf1.head(10)

Unmatched driver names (need manual fix): ['Nyck de VRIES' 'Guanyu ZHOU' nan]
StatsF1 records matched to session_key: 468 / 468


,finish_pos_statsf1,car_number_statsf1,driver_name_statsf1,team_statsf1,engine_statsf1,laps_statsf1,time_statsf1,points_statsf1,race_slug,round,driver_number,session_key
0,1,1.0,Max VERSTAPPEN,Red Bull,Honda RBPT,57.0,1h 33m 56.736s ( 196.862 km/h ),25.0,bahrein,1,1.0,7953
1,2,11.0,Sergio PEREZ,Red Bull,Honda RBPT,57.0,1h 34m 08.723s ( +11.987s ),18.0,bahrein,1,11.0,7953
2,3,14.0,Fernando ALONSO,Aston Martin,Mercedes,57.0,1h 34m 35.373s ( +38.637s ),15.0,bahrein,1,14.0,7953
3,4,55.0,Carlos SAINZ,Ferrari,Ferrari,57.0,1h 34m 44.788s ( +48.052s ),12.0,bahrein,1,55.0,7953
4,5,44.0,Lewis HAMILTON,Mercedes,Mercedes,57.0,1h 34m 47.713s ( +50.977s ),10.0,bahrein,1,44.0,7953
5,6,18.0,Lance STROLL,Aston Martin,Mercedes,57.0,1h 34m 51.238s ( +54.502s ),8.0,bahrein,1,18.0,7953
6,7,63.0,George RUSSELL,Mercedes,Mercedes,57.0,1h 34m 52.609s ( +55.873s ),6.0,bahrein,1,63.0,7953
7,8,77.0,Valtteri BOTTAS,Alfa Romeo,Ferrari,57.0,1h 35m 09.383s ( +1m 12.647s ),4.0,bahrein,1,77.0,7953
8,9,10.0,Pierre GASLY,Alpine,Renault,57.0,1h 35m 10.489s ( +1m 13.753s ),2.0,bahrein,1,10.0,7953
9,10,23.0,Alexander ALBON,Williams,Mercedes,57.0,1h 35m 26.510s ( +1m 29.774s ),1.0,bahrein,1,23.0,7953


We join the cleaned StatsF1 results into `df_combined` on `session_key` and `driver_number`. Because StatsF1 covers all 22 races, rows that OpenF1's `session_result` was missing will now have StatsF1 finishing positions and points available as an alternative source.

In [35]:
statsf1_keep = ["session_key", "driver_number", "round",
                "finish_pos_statsf1", "laps_statsf1", "time_statsf1",
                "points_statsf1", "team_statsf1"]
statsf1_keep = [c for c in statsf1_keep if c in df_statsf1.columns]
print(df_statsf1['driver_name_statsf1'])
print(df_combined['full_name'])

df_combined_full = df_combined.merge(
    df_statsf1[statsf1_keep],
    on=["session_key", "driver_number"],
    how="left"
)

print(f"Final combined dataset shape: {df_combined_full.shape}")
print(f"Columns: {list(df_combined_full.columns)}")
df_combined_full.head(10)

0       Max VERSTAPPEN
1         Sergio PEREZ
2      Fernando ALONSO
3         Carlos SAINZ
4       Lewis HAMILTON
            ...       
463        Guanyu ZHOU
464       Carlos SAINZ
465    Valtteri BOTTAS
466    Kevin MAGNUSSEN
467                NaN
Name: driver_name_statsf1, Length: 468, dtype: object
0                     NaN
1                     NaN
2                     NaN
3                     NaN
4                     NaN
              ...        
2271         Carlos SAINZ
2272    Gabriel BORTOLETO
2273      Alexander ALBON
2274         Pierre GASLY
2275     Franco COLAPINTO
Name: full_name, Length: 2276, dtype: object
Final combined dataset shape: (2276, 40)
Columns: ['position', 'driver_number', 'number_of_laps', 'points', 'dnf', 'dns', 'dsq', 'duration', 'gap_to_leader', 'meeting_key', 'session_key', 'session_name', 'circuit_short_name', 'country_name', 'date_start', 'full_name', 'team_name', 'country_code', 'total_laps', 'avg_lap_duration', 'fastest_lap', 'lap_time_std',

,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,...,avg_throttle,avg_brake,avg_rpm,drs_usage,round,finish_pos_statsf1,laps_statsf1,time_statsf1,points_statsf1,team_statsf1
0,1.0,1,57.0,26.0,False,False,False,5258.241,0,1208,...,NaN,NaN,NaN,NaN,5.0,1.0,57.0,1h 27m 38.241s ( 211.092 km/h ),26.0,Red Bull
1,2.0,11,57.0,18.0,False,False,False,5263.625,5.384,1208,...,NaN,NaN,NaN,NaN,5.0,2.0,57.0,1h 27m 43.625s ( +05.384s ),18.0,Red Bull
2,3.0,14,57.0,15.0,False,False,False,5284.546,26.305,1208,...,45.358393,19.366473,6394.527482,0.007281,5.0,3.0,57.0,1h 28m 04.546s ( +26.305s ),15.0,Aston Martin
3,4.0,63,57.0,12.0,False,False,False,5291.47,33.229,1208,...,47.658482,19.851412,6456.866830,0.024869,5.0,4.0,57.0,1h 28m 11.470s ( +33.229s ),12.0,Mercedes
4,5.0,55,57.0,10.0,False,False,False,5300.752,42.511,1208,...,48.735117,22.308540,6509.830768,0.038603,5.0,5.0,57.0,1h 28m 20.752s ( +42.511s ),10.0,Ferrari
5,6.0,44,57.0,8.0,False,False,False,5309.49,51.249,1208,...,NaN,NaN,NaN,NaN,5.0,6.0,57.0,1h 28m 29.490s ( +51.249s ),8.0,Mercedes
6,7.0,16,57.0,6.0,False,False,False,5311.229,52.988,1208,...,NaN,NaN,NaN,NaN,5.0,7.0,57.0,1h 28m 31.229s ( +52.988s ),6.0,Ferrari
7,8.0,10,57.0,4.0,False,False,False,5313.911,55.67,1208,...,NaN,NaN,NaN,NaN,5.0,8.0,57.0,1h 28m 33.911s ( +55.670s ),4.0,Alpine
8,9.0,31,57.0,2.0,False,False,False,5316.364,58.123,1208,...,NaN,NaN,NaN,NaN,5.0,9.0,57.0,1h 28m 36.364s ( +58.123s ),2.0,Alpine
9,10.0,20,57.0,1.0,False,False,False,5321.186,62.945,1208,...,NaN,NaN,NaN,NaN,5.0,10.0,57.0,1h 28m 41.186s ( +1m 02.945s ),1.0,Haas


In [36]:
nulls = df_combined_full["finish_pos_statsf1"].isna().sum()
total = len(df_combined_full)
print(f"Unmatched: {nulls} / {total} ({nulls/total:.1%})")

df_combined_full['finish_pos_statsf1'].sample(10)


Unmatched: 2167 / 2276 (95.2%)


1981    NaN
124     NaN
604     NaN
75      NaN
2213    NaN
1181    NaN
1572    NaN
1756    NaN
42      NaN
539     NaN
Name: finish_pos_statsf1, dtype: object